In [ ]:
#| echo: false
#| fig-cap: "Left: analyzers. Middle: four overlaps per phi (colors match analyzers). Right: Calculated S(phi).  Far-right: S(phi)"
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

# --------------------------
# Basic setup
# --------------------------
N  = 2000
t  = np.linspace(0, 2*np.pi, N, endpoint=False)
dt = t[1] - t[0]
den = np.trapezoid(np.cos(t)**2, t)  # ∫_0^{2π} cos^2 dt = π
print(den)

# Analyzer colors / styles
COL_A,  LS_A  = "C0", "-"
COL_AP, LS_AP = "C0", "--"
COL_B,  LS_B  = "C1", "-"
COL_BP, LS_BP = "C2", "-"

COL_PROD = "0.5"   # gray
COL_CUM  = "0.0"   # black

# --------------------------
# Helpers
# --------------------------
def corr_plots(ax, a, b, who, show_legend=False):
    """
    who = tuple like ('a','b') or ('a','b′') etc. Used to color-match x,y to analyzers.
    """
    x = np.cos(t - a)
    y = np.cos(t - b)
    prod = x * y

    # normalized full-period correlation in [-1,1]
    E_num = np.trapezoid(prod, t) / den

    # cumulative normalized integral (running average-like)
    cum = np.cumsum(prod) * dt / den
    tt  = np.insert(t, 0, 0.0)
    cum = np.insert(cum, 0, 0.0)

    # choose colors/linestyles from "who"
    who_x, who_y = who

    if   who_x == "a":  col_x, ls_x = COL_A,  LS_A
    elif who_x == "a′": col_x, ls_x = COL_AP, LS_AP
    else:               col_x, ls_x = "C7",  "-"

    if   who_y == "b":  col_y, ls_y = COL_B,  LS_B
    elif who_y == "b′": col_y, ls_y = COL_BP, LS_BP
    else:               col_y, ls_y = "C7",   "-"

    # plots
    ax.plot(t, x,  lw=1.2, label=f"x=cos(t-{who_x})", color=col_x, linestyle=ls_x)
    ax.plot(t, y,  lw=1.1, label=f"y=cos(t-{who_y})", color=col_y, linestyle=ls_y, alpha=0.95)
    ax.plot(t, prod, lw=0.9, label="product x·y", color=COL_PROD, alpha=0.85)
    ax.plot(tt, cum, lw=1.2, ls="--", label="cumulative normalized overlap", color=COL_CUM)

    # cosmetics
    ax.set_ylim(-1.6, 1.6)
    ax.set_xlim(0, 2*np.pi)
    ax.grid(True, alpha=0.2, linewidth=0.6)
    ax.tick_params(labelsize=7)
    for spine in ax.spines.values():
        spine.set_alpha(0.4)
        spine.set_linewidth(0.6)

    if show_legend:
        ax.legend(loc="upper right", fontsize=7, frameon=False)

    return E_num

def plot_vectors_left(ax, phi):
    ax.clear()
    th = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(th), np.sin(th), lw=1, color="0.25")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.grid(True, alpha=0.2, linewidth=0.6)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    a, ap = 0.0, np.pi/2
    b, bp = +phi/2.0, -phi/2.0

    def vec(angle):
        return np.array([np.cos(angle), np.sin(angle)])

    arrows = [("a", a, COL_A, LS_A), ("a′", ap, COL_AP, LS_AP),
              ("b", b, COL_B, LS_B), ("b′", bp, COL_BP, LS_BP)]
    for label, ang, col, lstyle in arrows:
        v = vec(ang)
        ax.arrow(0, 0, v[0], v[1],
                 head_width=0.06, head_length=0.08,
                 length_includes_head=True,
                 color=col, linestyle=lstyle, lw=1.0)
        label="" # comment out this line to plot the a / a' / b / b' vector labels
        ax.text(1.06*v[0], 1.06*v[1], label, fontsize=9,
                ha="center", va="center", color=col)

def plot_S_right(ax, S):
    ax.clear()
    ax.axis("off")
    # ax.text(0.5, 0.62, r"$S$", fontsize=14, ha="center", va="center")
    ax.text(0.5, 0.40, f"{S:1.2f}", fontsize=10, ha="center", va="center")
    # ax.text(0.5, 0.17,
            # rf"(classical 2, Tsirelson {2*np.sqrt(2):.5f})",
            # fontsize=8.5, ha="center", va="center", color="0.3")

def plot_phi_left(ax, phi):
    ax.clear()
    ax.axis("off")
    # One or two lines; pick your favorite
    # ax.text(0.5, 0.55, r"$\varphi$", fontsize=12, ha="center", va="center")
    ax.text(0.5, 0.50, f"{np.degrees(phi):.0f}°", fontsize=10, ha="center", va="center")
    # If you also want radians, add this line (or swap it in):
    # ax.text(0.5, 0.10, f"{phi:.3f} rad", fontsize=8.5, ha="center", va="center", color="0.35")


# --------------------------
# φ values (kept as-is, but you can thin rows via ROW_STRIDE)
# --------------------------

step_deg = 45
cycles = 2
phis_deg = np.arange(0, cycles * 360 + step_deg, step_deg)
phis = (np.pi / 180) * phis_deg

ROW_STRIDE = 1  # set to 2 (or 3) to thin rows without changing 'phis'
phis_plot = phis[::ROW_STRIDE]
nrows = len(phis_plot)

# --------------------------
# Layout
# --------------------------
# Smaller per-row height + tighter spacing
per_row_h = 0.6   # << shrink this to reduce total figure height
fig_w     = 10
fig_h     = max(3.6, per_row_h * nrows)

fig = plt.figure(figsize=(fig_w, fig_h))
# fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=True)
gs  = gridspec.GridSpec(
    nrows, 8, figure=fig,
    #    φ     vectors   E(a,b) ...             S     S-curve
    width_ratios=[0.8,   1.05,   1.8, 1.8, 1.8, 1.8, 0.9,  5.0],
    wspace=0.18, hspace=0.22
)

S_values = []

for i, phi in enumerate(phis_plot):
    # New left numeric φ (col 0)
    ax_phi = fig.add_subplot(gs[i, 0])
    plot_phi_left(ax_phi, phi)
    if i == 0:
        ax_phi.set_title("φ", fontsize=10, pad=6)

    # Left vectors (now col 1)
    ax_vec = fig.add_subplot(gs[i, 1])
    plot_vectors_left(ax_vec, phi)
    if i == 0:
        ax_vec.set_title("Analyzers", fontsize=10, pad=6)

    # CHSH correlation settings (no doubled angles)
    a, ap = 0.0, np.pi/2
    b, bp = +phi/2.0, -phi/2.0

    # Four correlation panels (now cols 2..5)
    ax_Eab   = fig.add_subplot(gs[i, 2], sharex=None,    sharey=None)
    ax_Eabp  = fig.add_subplot(gs[i, 3], sharex=ax_Eab,  sharey=ax_Eab)
    ax_Eapb  = fig.add_subplot(gs[i, 4], sharex=ax_Eab,  sharey=ax_Eab)
    ax_Eapbp = fig.add_subplot(gs[i, 5], sharex=ax_Eab,  sharey=ax_Eab)

    if i == 0:
        ax_Eab.set_title("E(a,b)",     fontsize=10, pad=6)
        ax_Eabp.set_title("E(a,b′)",   fontsize=10, pad=6)
        ax_Eapb.set_title("E(a′,b)",   fontsize=10, pad=6)
        ax_Eapbp.set_title("E(a′,b′)", fontsize=10, pad=6)

    E_ab   = corr_plots(ax_Eab,   a,  b,  ('a','b'),   show_legend=False)
    E_abp  = corr_plots(ax_Eabp,  a,  bp, ('a','b′'))
    E_apb  = corr_plots(ax_Eapb,  ap, b,  ('a′','b'))
    E_apbp = corr_plots(ax_Eapbp, ap, bp, ('a′','b′'))

    # Clean y labels on the latter three
    for ax in (ax_Eabp, ax_Eapb, ax_Eapbp):
        ax.set_yticklabels([])
        ax.set_ylabel("")
    if i < nrows - 1:
        for ax in (ax_Eab, ax_Eabp, ax_Eapb, ax_Eapbp):
            ax.set_xticklabels([])

    # (Optional) remove row_title since φ is now shown in its own column
    # -- delete or comment out the block that writes row_title on ax_Eab --

    # CHSH S for this row
    S = E_ab + E_abp + E_apb - E_apbp
    S_values.append(S)

    # Right numeric S (now col 6)
    ax_num = fig.add_subplot(gs[i, 6])
    plot_S_right(ax_num, S)
    if i == 0:
        ax_num.set_title(r"$S(\varphi)$", fontsize=10, pad=6)


# --- Far-right tall plot: spans all rows, now column 7 ---
ax_Scurve = fig.add_subplot(gs[:, 7])

phi_min, phi_max = min(phis), max(phis)
phi_grid   = np.linspace(phi_min, phi_max, 800)
theta_grid = phi_grid / 2.0
S_curve    = 2*(np.cos(theta_grid) + np.sin(theta_grid))

# Map φ to a row-aligned vertical coordinate y ∈ [0, nrows-1]
def phi_to_rowy(phi):
    span = (phi_max - phi_min) if (phi_max > phi_min) else 1.0
    return (phi - phi_min) / span * (nrows - 1)

y_curve = phi_to_rowy(phi_grid)
y_pts   = [phi_to_rowy(p) for p in phis_plot]

ax_Scurve.scatter(S_values, y_pts, color="black", zorder=3, s=10, label="computed S")
ax_Scurve.plot(S_curve, y_curve, lw=2,
               label=r"$S(\varphi)=2\!\left(\cos\frac{\varphi}{2}+\sin\frac{\varphi}{2}\right)$", color="#808080")
# ax_Scurve.plot(S_curve, y_curve, lw=2,
            #    label=r"$S(\varphi)$", color="#808080")
ax_Scurve.axvline(2, linestyle="--", color="red",  label="classical = 2", linewidth=1)
ax_Scurve.axvline(-2, linestyle="--", color="red", linewidth=1)
ax_Scurve.axvline(2*np.sqrt(2), linestyle="-", color="red",
                  linewidth=1)
ax_Scurve.axvline(-2*np.sqrt(2), linestyle="-", color="red",
                  label=r"Tsirelson $2\sqrt{2}$", linewidth=1)

ax_Scurve.set_ylim(-0.5, nrows - 0.5)
ax_Scurve.invert_yaxis()
ax_Scurve.set_yticks(y_pts)
# ax_Scurve.set_yticklabels([f"φ={np.degrees(p):.0f}°" for p in phis_plot], fontsize=8)
ax_Scurve.set_yticklabels([])
ax_Scurve.set_xlabel("S", fontsize=10)
ax_Scurve.set_xticks([x for x in range(-3, 4)])
ax_Scurve.set_title("S", fontsize=10, pad=6)
ax_Scurve.grid(True, axis="y", alpha=0.25, linewidth=0.7)
ax_Scurve.grid(True, axis="x", alpha=0.7, linewidth=0.7)

ax_Scurve.set_xlim(-3.25, 3.25)
ax_Scurve.legend(loc="upper center", fontsize=8, frameon=False)

# After you've added all artists:
handles, labels = ax_Scurve.get_legend_handles_labels()
# fig.legend(handles, labels,
        #    loc="upper left", bbox_to_anchor=(1.05, 1.0),
        #    frameon=False, fontsize=8)

# fig.suptitle(
    # "Left: analyzers.  Middle: four overlaps per φ (colors match analyzers).  "
    # "Right: numeric S.  Far-right: S(φ) curve",
    # y=0.995, fontsize=13
# )
# plt.tight_layout(rect=(0, 0, 1, 0.985))
fig.subplots_adjust(
    left=0.035, right=0.995, bottom=0.03, top=0.985,
    wspace=0.18, hspace=0.22
)

plt.savefig('correlation_plot.png')
plt.show()
fig.set_dpi(300)  # or plt.rcParams['figure.dpi'] = 200


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Spin-1/2 CHSH target settings (standard symmetric choice)
# a = 0°, a' = 90°, b = +45°, b' = -45°  -> |S| = 2√2 (quantum)
# ----------------------------
a  = 0.0
ap = np.pi/2
phi = np.pi/2
b  = +phi/2
bp = -phi/2

pairs = [
    ("E(a,b)",   a,  b),
    ("E(a,b′)",  a,  bp),
    ("E(a′,b)",  ap, b),
    ("E(a′,b′)", ap, bp),
]

def E_quantum_spin(a1, b1):
    # Spin-1/2 singlet: E(a,b) = -cos(a-b)
    return -np.cos(a1 - b1)

def quadrant_probs_from_E(E):
    # with unbiased marginals: P++=P--=(1+E)/4 ; P+-=P-+=(1-E)/4
    Ppp = (1 + E)/4
    Pmm = (1 + E)/4
    Ppm = (1 - E)/4
    Pmp = (1 - E)/4
    return Ppp, Ppm, Pmp, Pmm

# ----------------------------
# Torus texture (intuition): relative-angle diagonal bands
# We'll reuse the "unwrapped torus" texture and SHIFT coordinates so that:
#   Alice "+" bin is left half (x in [0,π))
#   Bob   "+" bin is bottom half (y in [0,π))
# for each measurement pair.
# ----------------------------
M = 420
thetaA = np.linspace(0, 2*np.pi, M, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, M, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")

# Diagonal pair texture (for intuition only)
W = np.cos((TA - TB)/2.0)**2
Wn = (W - W.min()) / (W.max() - W.min() + 1e-12)

def shift_to_halfspace_bins(Wgrid, a_setting, b_setting):
    # Define "+" bins as θ in [setting-π/2, setting+π/2). Shift so those intervals become [0,π).
    shiftA = (a_setting - np.pi/2) % (2*np.pi)
    shiftB = (b_setting - np.pi/2) % (2*np.pi)
    kA = int(round(shiftA / (2*np.pi) * Wgrid.shape[1]))
    kB = int(round(shiftB / (2*np.pi) * Wgrid.shape[0]))
    W2 = np.roll(Wgrid, -kA, axis=1)
    W2 = np.roll(W2, -kB, axis=0)
    return W2

# ----------------------------
# Build the combined figure
# ----------------------------
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.0, 1.15], wspace=0.25, hspace=0.22)

# 2x2 torus panels
axes_torus = [
    fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]),
]

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

last_im = None
for ax, (title, a1, b1) in zip(axes_torus, pairs):
    W2 = shift_to_halfspace_bins(Wn, a1, b1)

    last_im = ax.imshow(
        W2,
        origin="lower",
        extent=(0, 2*np.pi, 0, 2*np.pi),
        interpolation="nearest",
        aspect="equal",
    )
    ax.axvline(np.pi, linewidth=2)  # Alice +/− boundary in shifted coords
    ax.axhline(np.pi, linewidth=2)  # Bob   +/− boundary in shifted coords

    E = E_quantum_spin(a1, b1)
    Ppp, Ppm, Pmp, Pmm = quadrant_probs_from_E(E)

    # annotate quadrants (centers)
    centers = {
        "++": (np.pi/2, np.pi/2),
        "+-": (3*np.pi/2, np.pi/2),
        "-+": (np.pi/2, 3*np.pi/2),
        "--": (3*np.pi/2, 3*np.pi/2),
    }
    vals = {"++": Ppp, "+-": Ppm, "-+": Pmp, "--": Pmm}

    for key, (cx, cy) in centers.items():
        p = vals[key]
        ax.text(
            cx, cy,
            f"{key}\nP={p:.3f}\nΔP={p-0.25:+.3f}",
            ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round", alpha=0.75),
        )

    ax.set_title(f"{title}\nΔ={np.degrees(a1-b1):+.1f}°,  E={E:+.3f}", fontsize=11)
    ax.set_xticks(ticks, ticklabels)
    ax.set_yticks(ticks, ticklabels)

# Shared colorbar for the torus texture (intuition)
cax = fig.add_axes([0.62, 0.12, 0.015, 0.76])
cbar = fig.colorbar(last_im, cax=cax)
cbar.set_label("Diagonal texture (intuition)", fontsize=10)

# ----------------------------
# Right column: analyzer vectors + S(φ) curve + summary
# ----------------------------
ax_vec = fig.add_subplot(gs[0, 2])
th = np.linspace(0, 2*np.pi, 500)
ax_vec.plot(np.cos(th), np.sin(th), lw=1)
ax_vec.set_aspect("equal", adjustable="box")
ax_vec.set_xlim(-1.15, 1.15)
ax_vec.set_ylim(-1.15, 1.15)
ax_vec.grid(True, alpha=0.25)
ax_vec.set_xticks([]); ax_vec.set_yticks([])
for spine in ax_vec.spines.values():
    spine.set_visible(False)

def arrow(ax, ang, label):
    v = np.array([np.cos(ang), np.sin(ang)])
    ax.arrow(0, 0, v[0], v[1], head_width=0.06, head_length=0.08,
             length_includes_head=True, lw=1.5)
    ax.text(1.06*v[0], 1.06*v[1], label, ha="center", va="center", fontsize=11)

arrow(ax_vec, a,  "a")
arrow(ax_vec, ap, "a′")
arrow(ax_vec, b,  "b")
arrow(ax_vec, bp, "b′")
ax_vec.set_title("Spin-1/2 CHSH analyzer axes", fontsize=12)

# Bottom-right: S(φ) curve
ax_S = fig.add_subplot(gs[1, 2])

phi_grid = np.linspace(0, 2*np.pi, 800)
S_grid = (
    E_quantum_spin(0.0, +phi_grid/2) +
    E_quantum_spin(0.0, -phi_grid/2) +
    E_quantum_spin(np.pi/2, +phi_grid/2) -
    E_quantum_spin(np.pi/2, -phi_grid/2)
)
ax_S.plot(S_grid, np.degrees(phi_grid), lw=2)
ax_S.axvline(2, linestyle="--", linewidth=1)
ax_S.axvline(-2, linestyle="--", linewidth=1)
ax_S.axvline(2*np.sqrt(2), linewidth=1)
ax_S.axvline(-2*np.sqrt(2), linewidth=1)
ax_S.grid(True, alpha=0.35)
ax_S.set_xlabel("S", fontsize=11)
ax_S.set_ylabel(r"$\varphi$ (deg)", fontsize=11)
ax_S.set_title("Quantum S(φ) for spin-1/2 singlet", fontsize=12)

# Mark the chosen φ
S_phi = (
    E_quantum_spin(a, b) +
    E_quantum_spin(a, bp) +
    E_quantum_spin(ap, b) -
    E_quantum_spin(ap, bp)
)
ax_S.scatter([S_phi], [np.degrees(phi)], s=40, zorder=3)
ax_S.text(S_phi, np.degrees(phi), f"  φ={np.degrees(phi):.0f}°, S={S_phi:+.3f}", va="center", fontsize=10)

# Summary text placed between the two right panels
summary = (
    "Settings: a=0°, a′=90°, b=+45°, b′=−45°\n"
    "Spin singlet: E(a,b) = −cos(a−b)\n\n"
    f"E(a,b)   = {E_quantum_spin(a,b):+.3f}\n"
    f"E(a,b′)  = {E_quantum_spin(a,bp):+.3f}\n"
    f"E(a′,b)  = {E_quantum_spin(ap,b):+.3f}\n"
    f"E(a′,b′) = {E_quantum_spin(ap,bp):+.3f}\n\n"
    "S = E(a,b)+E(a,b′)+E(a′,b)−E(a′,b′)\n"
    f"S = {S_phi:+.3f}   (|S| max = {2*np.sqrt(2):.3f})"
)
fig.text(0.69, 0.50, summary, fontsize=11, va="center")

fig.suptitle("Combined figure (spin-1/2): torus bins ↔ quadrant weights ↔ CHSH correlations", fontsize=14, y=0.98)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ----------------------------
# Spin-1/2 CHSH torus animation
# ----------------------------
def E_quantum_spin(a1, b1):
    # Spin-1/2 singlet: E(a,b) = -cos(a-b)
    return -np.cos(a1 - b1)

def quadrant_probs_from_E(E):
    # unbiased marginals
    Ppp = (1 + E)/4
    Pmm = (1 + E)/4
    Ppm = (1 - E)/4
    Pmp = (1 - E)/4
    return Ppp, Ppm, Pmp, Pmm

def shift_to_halfspace_bins(Wgrid, a_setting, b_setting):
    # Define "+" bins as θ in [setting-π/2, setting+π/2). Shift so those intervals become [0,π).
    shiftA = (a_setting - np.pi/2) % (2*np.pi)
    shiftB = (b_setting - np.pi/2) % (2*np.pi)
    kA = int(round(shiftA / (2*np.pi) * Wgrid.shape[1]))
    kB = int(round(shiftB / (2*np.pi) * Wgrid.shape[0]))
    W2 = np.roll(Wgrid, -kA, axis=1)
    W2 = np.roll(W2, -kB, axis=0)
    return W2

# Base torus texture (intuition background)
M = 240
thetaA = np.linspace(0, 2*np.pi, M, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, M, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")
W = np.cos((TA - TB)/2.0)**2
Wn = (W - W.min()) / (W.max() - W.min() + 1e-12)

# CHSH fixed A settings
a  = 0.0
ap = np.pi/2

# Precompute S(φ) curve for the right panel
phi_grid = np.linspace(0, 2*np.pi, 900)
S_grid = (
    E_quantum_spin(0.0, +phi_grid/2) +
    E_quantum_spin(0.0, -phi_grid/2) +
    E_quantum_spin(np.pi/2, +phi_grid/2) -
    E_quantum_spin(np.pi/2, -phi_grid/2)
)

# Figure layout: 2x2 torus panels + S(φ) curve
fig = plt.figure(figsize=(12, 7))
gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.0, 1.15], wspace=0.25, hspace=0.22)

axes_torus = [
    fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]),
]
ax_S = fig.add_subplot(gs[:, 2])

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

pair_names = ["E(a,b)", "E(a,b′)", "E(a′,b)", "E(a′,b′)"]

# Initialize artists for torus panels
ims = []
quad_texts = []  # list of lists, one per panel
title_texts = []

for ax, name in zip(axes_torus, pair_names):
    # placeholder initial phi
    phi0 = 0.0
    b0, bp0 = +phi0/2, -phi0/2
    if name == "E(a,b)":
        a1, b1 = a, b0
    elif name == "E(a,b′)":
        a1, b1 = a, bp0
    elif name == "E(a′,b)":
        a1, b1 = ap, b0
    else:
        a1, b1 = ap, bp0

    W2 = shift_to_halfspace_bins(Wn, a1, b1)
    im = ax.imshow(
        W2,
        origin="lower",
        extent=(0, 2*np.pi, 0, 2*np.pi),
        interpolation="nearest",
        aspect="equal",
    )
    ims.append(im)

    ax.axvline(np.pi, linewidth=2)
    ax.axhline(np.pi, linewidth=2)
    ax.set_xticks(ticks, ticklabels)
    ax.set_yticks(ticks, ticklabels)

    centers = {
        "++": (np.pi/2, np.pi/2),
        "+-": (3*np.pi/2, np.pi/2),
        "-+": (np.pi/2, 3*np.pi/2),
        "--": (3*np.pi/2, 3*np.pi/2),
    }
    tlist = []
    for key, (cx, cy) in centers.items():
        txt = ax.text(
            cx, cy,
            f"{key}\nP=0.250\nΔP=+0.000",
            ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round", alpha=0.75),
        )
        tlist.append((key, txt))
    quad_texts.append(tlist)

    title = ax.set_title(f"{name}", fontsize=11)
    title_texts.append(title)

# Colorbar for torus texture
cax = fig.add_axes([0.62, 0.12, 0.015, 0.76])
cbar = fig.colorbar(ims[0], cax=cax)
cbar.set_label("Diagonal texture (intuition)", fontsize=10)

# S(φ) curve panel (static curve, moving marker)
ax_S.plot(S_grid, np.degrees(phi_grid), lw=2)
ax_S.axvline(2, linestyle="--", linewidth=1)
ax_S.axvline(-2, linestyle="--", linewidth=1)
ax_S.axvline(2*np.sqrt(2), linewidth=1)
ax_S.axvline(-2*np.sqrt(2), linewidth=1)
ax_S.grid(True, alpha=0.35)
ax_S.set_xlabel("S", fontsize=11)
ax_S.set_ylabel(r"$\varphi$ (deg)", fontsize=11)
ax_S.set_title("Quantum S(φ) for spin-1/2 singlet", fontsize=12)

marker = ax_S.scatter([S_grid[0]], [np.degrees(phi_grid[0])], s=50, zorder=3)
marker_label = ax_S.text(S_grid[0], np.degrees(phi_grid[0]), "  ", va="center", fontsize=10)

fig.suptitle("Spin-1/2 CHSH: quadrant mass shifts on the joint-angle torus as φ sweeps", fontsize=14, y=0.98)

# Animation sweep
nframes = 90
phis = np.linspace(0, 2*np.pi, nframes, endpoint=False)

def update(frame_idx):
    phi = phis[frame_idx]
    b = +phi/2
    bp = -phi/2

    settings = [
        (a, b),
        (a, bp),
        (ap, b),
        (ap, bp),
    ]

    for ax_i, (a1, b1) in enumerate(settings):
        W2 = shift_to_halfspace_bins(Wn, a1, b1)
        ims[ax_i].set_data(W2)

        E = E_quantum_spin(a1, b1)
        Ppp, Ppm, Pmp, Pmm = quadrant_probs_from_E(E)
        vals = {"++": Ppp, "+-": Ppm, "-+": Pmp, "--": Pmm}

        for key, txt in quad_texts[ax_i]:
            p = vals[key]
            txt.set_text(f"{key}\nP={p:.3f}\nΔP={p-0.25:+.3f}")

        title_texts[ax_i].set_text(
            f"{pair_names[ax_i]}\nΔ={np.degrees(a1-b1):+.1f}°,  E={E:+.3f}"
        )

    S_phi = (
        E_quantum_spin(a, b) +
        E_quantum_spin(a, bp) +
        E_quantum_spin(ap, b) -
        E_quantum_spin(ap, bp)
    )
    y = np.degrees(phi)
    marker.set_offsets(np.array([[S_phi, y]]))
    marker_label.set_position((S_phi, y))
    marker_label.set_text(f"  φ={y:.0f}°, S={S_phi:+.3f}")

    return []  # blit disabled

anim = FuncAnimation(fig, update, frames=nframes, interval=90, blit=False)

out_path = "spin_chsh_torus_animation.gif"
anim.save(out_path, writer=PillowWriter(fps=12))

out_path


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

# ----------------------------
# Animation: single torus panel showing how quadrant masses shift with Δ
# Spin-1/2 singlet: E(Δ) = -cos(Δ)
# ----------------------------
def E_spin(delta):
    return -np.cos(delta)

def quadrant_probs_from_E(E):
    # unbiased marginals
    Ppp = (1 + E)/4
    Pmm = (1 + E)/4
    Ppm = (1 - E)/4
    Pmp = (1 - E)/4
    return Ppp, Ppm, Pmp, Pmm

def shift_to_halfspace_bins(Wgrid, a_setting, b_setting):
    # Define "+" bins as θ in [setting-π/2, setting+π/2). Shift so those intervals become [0,π).
    shiftA = (a_setting - np.pi/2) % (2*np.pi)
    shiftB = (b_setting - np.pi/2) % (2*np.pi)
    kA = int(round(shiftA / (2*np.pi) * Wgrid.shape[1]))
    kB = int(round(shiftB / (2*np.pi) * Wgrid.shape[0]))
    W2 = np.roll(Wgrid, -kA, axis=1)
    W2 = np.roll(W2, -kB, axis=0)
    return W2

# Base diagonal texture (intuition background)
M = 180
thetaA = np.linspace(0, 2*np.pi, M, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, M, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")
W = np.cos((TA - TB)/2.0)**2
Wn = (W - W.min()) / (W.max() - W.min() + 1e-12)

# Sweep Δ from 0..180° (0..π)
nframes = 48
deltas = np.linspace(0, np.pi, nframes)

# Figure: torus panel + E(Δ) curve
fig = plt.figure(figsize=(10, 5.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.05], wspace=0.22)

ax_t = fig.add_subplot(gs[0, 0])
ax_e = fig.add_subplot(gs[0, 1])

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

# Initialize torus panel at Δ=0
a = 0.0
b = deltas[0]

im = ax_t.imshow(
    shift_to_halfspace_bins(Wn, a, b),
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    interpolation="nearest",
    aspect="equal",
)
ax_t.axvline(np.pi, linewidth=2)
ax_t.axhline(np.pi, linewidth=2)
ax_t.set_xticks(ticks, ticklabels)
ax_t.set_yticks(ticks, ticklabels)
ax_t.set_xlabel(r"$\theta_A$ (shifted)")
ax_t.set_ylabel(r"$\theta_B$ (shifted)")

# Quadrant texts
centers = {
    "++": (np.pi/2, np.pi/2),
    "+-": (3*np.pi/2, np.pi/2),
    "-+": (np.pi/2, 3*np.pi/2),
    "--": (3*np.pi/2, 3*np.pi/2),
}
txt_objs = {}
for key, (cx, cy) in centers.items():
    txt_objs[key] = ax_t.text(
        cx, cy, "",
        ha="center", va="center", fontsize=10,
        bbox=dict(boxstyle="round", alpha=0.75),
    )

title_t = ax_t.set_title("", fontsize=12)

cbar = fig.colorbar(im, ax=ax_t, fraction=0.046, pad=0.04)
cbar.set_label("Diagonal texture (intuition)")

# E(Δ) curve panel
E_curve = E_spin(deltas)
ax_e.plot(np.degrees(deltas), E_curve, lw=2)
ax_e.grid(True, alpha=0.35)
ax_e.set_xlim(0, 180)
ax_e.set_ylim(-1.05, 1.05)
ax_e.set_xlabel(r"$\Delta$ (deg)")
ax_e.set_ylabel(r"$E(\Delta)$")
ax_e.set_title(r"Spin-1/2 singlet: $E(\Delta)=-\cos\Delta$", fontsize=12)

marker = ax_e.scatter([0], [E_curve[0]], s=60, zorder=3)
marker_label = ax_e.text(0, E_curve[0], "  ", va="center", fontsize=10)

fig.suptitle("How the joint probability mass moves between quadrants as Δ changes", fontsize=14, y=0.98)

def update(i):
    delta = deltas[i]
    b = delta  # set b=a+Δ with a=0

    # Update torus background (shifted so + bins are left/bottom halves)
    im.set_data(shift_to_halfspace_bins(Wn, a, b))

    # Quantum probabilities from E
    E = E_spin(delta)
    Ppp, Ppm, Pmp, Pmm = quadrant_probs_from_E(E)
    vals = {"++": Ppp, "+-": Ppm, "-+": Pmp, "--": Pmm}

    for key in ["++", "+-", "-+", "--"]:
        p = vals[key]
        txt_objs[key].set_text(f"{key}\nP={p:.3f}\nΔP={p-0.25:+.3f}")

    title_t.set_text(f"Δ={np.degrees(delta):.0f}°   E={E:+.3f}  = (P++ + P--) − (P+- + P-+)")

    # Update E marker
    xdeg = np.degrees(delta)
    marker.set_offsets(np.array([[xdeg, E]]))
    marker_label.set_position((xdeg, E))
    marker_label.set_text(f"  Δ={xdeg:.0f}°, E={E:+.3f}")

    return []  # blit off

anim = FuncAnimation(fig, update, frames=nframes, interval=90, blit=False)

out_mp4 = "torus_quadrant_mass_spin_singlet.mp4"
writer = FFMpegWriter(fps=12, bitrate=1200)
anim.save(out_mp4, writer=writer)

out_mp4


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Two "torus models" side-by-side
#   Left: Local hidden-variable sign model (triangle-wave correlation)
#   Right: "Cheating" setting-dependent torus density that reproduces quantum singlet E=-cosΔ
# ----------------------------

rng = np.random.default_rng(0)

def wrap2pi(x):
    return np.mod(x, 2*np.pi)

def E_quantum_spin(delta):
    return -np.cos(delta)

def E_local_sign(delta, nsamp=200000):
    # A = sign(cos(λ)), B = -sign(cos(λ+delta))  (a=0, b=-delta)
    lam = rng.uniform(0, 2*np.pi, size=nsamp)
    A = np.sign(np.cos(lam)); A[A == 0] = 1
    B = -np.sign(np.cos(lam + delta)); B[B == 0] = 1
    return float(np.mean(A * B))

def quadrant_probs_from_E(E):
    # unbiased marginals
    Ppp = (1 + E) / 4
    Pmm = (1 + E) / 4
    Ppm = (1 - E) / 4  # (+,-)
    Pmp = (1 - E) / 4  # (-,+)
    return Ppp, Ppm, Pmp, Pmm

# Choose a representative Δ (where the CHSH "0.707" shows up)
delta = np.pi/4  # 45°
deg = 180/np.pi

# Analyzer settings for this slice (we can set a=0, b=delta without loss of generality)
a = 0.0
b = delta

# Shift to "halfspace bin" coordinates so + is always left/bottom half:
# x = (θA + π/2) mod 2π, y = (θB + π/2) mod 2π
def to_halfspace_coords(thetaA, thetaB):
    x = wrap2pi(thetaA + np.pi/2)
    y = wrap2pi(thetaB + np.pi/2)
    return x, y

# ----------------------------
# LEFT MODEL: Local sign hidden-variable model
# Interpret λ as a shared phase; outcomes are deterministic given λ and local settings.
# Build a torus "density" by mapping λ -> (θA, θB) along a diagonal (smear it for visibility).
# ----------------------------
nsamp_density = 250000
lam = rng.uniform(0, 2*np.pi, size=nsamp_density)

thetaA = wrap2pi(lam - a)
thetaB = wrap2pi(lam - b)

# Add small jitter so the diagonal becomes a visible band (purely for visualization)
j = 0.12
thetaA_j = wrap2pi(thetaA + rng.normal(0, j, size=nsamp_density))
thetaB_j = wrap2pi(thetaB + rng.normal(0, j, size=nsamp_density))

xL, yL = to_halfspace_coords(thetaA_j, thetaB_j)

# 2D histogram on [0,2π)×[0,2π)
bins = 180
H_L, xe, ye = np.histogram2d(xL, yL, bins=bins, range=[[0, 2*np.pi], [0, 2*np.pi]], density=True)

# Quadrant probabilities from this density (approx) by integrating histogram cells
# Each bin cell has area:
dx = (2*np.pi)/bins
dy = (2*np.pi)/bins
cell_area = dx * dy

# Quadrant index split at π
split = bins // 2
Ppp_L = H_L[:split, :split].sum() * cell_area
Ppm_L = H_L[split:, :split].sum() * cell_area
Pmp_L = H_L[:split, split:].sum() * cell_area
Pmm_L = H_L[split:, split:].sum() * cell_area

E_L_from_quads = (Ppp_L + Pmm_L) - (Ppm_L + Pmp_L)

# Also compute the "true" E for the sign model directly from λ (faster, more accurate)
E_L = E_local_sign(delta, nsamp=300000)

# ----------------------------
# RIGHT MODEL: "Cheating" setting-dependent torus density
# Make the density uniform within each quadrant so that quadrant integrals match the quantum probabilities.
# ----------------------------
E_Q = E_quantum_spin(delta)
Ppp_Q, Ppm_Q, Pmp_Q, Pmm_Q = quadrant_probs_from_E(E_Q)

# Build a grid density ρ_Q(x,y) piecewise-constant by quadrant
grid = 240
x = np.linspace(0, 2*np.pi, grid, endpoint=False)
y = np.linspace(0, 2*np.pi, grid, endpoint=False)
X, Y = np.meshgrid(x, y, indexing="xy")

rho_Q = np.zeros_like(X, dtype=float)
# quadrant areas are π^2, so density = P / π^2 inside each quadrant
area_q = (np.pi * np.pi)

mask_xp = X < np.pi
mask_yp = Y < np.pi
# ++ (bottom-left)
rho_Q[mask_xp & mask_yp] = Ppp_Q / area_q
# +- (bottom-right): x>=π, y<π
rho_Q[(~mask_xp) & mask_yp] = Ppm_Q / area_q
# -+ (top-left): x<π, y>=π
rho_Q[mask_xp & (~mask_yp)] = Pmp_Q / area_q
# -- (top-right): x>=π, y>=π
rho_Q[(~mask_xp) & (~mask_yp)] = Pmm_Q / area_q

# Verify by numerical integration on the grid (Riemann sum)
dxg = (2*np.pi)/grid
dyg = (2*np.pi)/grid
cell_area_g = dxg * dyg

Ppp_Q_num = rho_Q[mask_xp & mask_yp].sum() * cell_area_g
Ppm_Q_num = rho_Q[(~mask_xp) & mask_yp].sum() * cell_area_g
Pmp_Q_num = rho_Q[mask_xp & (~mask_yp)].sum() * cell_area_g
Pmm_Q_num = rho_Q[(~mask_xp) & (~mask_yp)].sum() * cell_area_g
E_Q_from_quads = (Ppp_Q_num + Pmm_Q_num) - (Ppm_Q_num + Pmp_Q_num)

# ----------------------------
# Bottom row: E(Δ) curves + CHSH S for standard angles
# ----------------------------
deltas = np.linspace(0, np.pi, 200)
Ecurve_Q = -np.cos(deltas)
Ecurve_L = np.array([E_local_sign(d, nsamp=60000) for d in deltas[::5]])  # subsample for speed
deltas_L = deltas[::5]

# CHSH S at a=0, a'=π/2, b=+π/4, b'=-π/4
a0, ap0 = 0.0, np.pi/2
b0, bp0 = np.pi/4, -np.pi/4
def S_from_Efunc(Efunc):
    return (
        Efunc(a0 - b0) + Efunc(a0 - bp0) + Efunc(ap0 - b0) - Efunc(ap0 - bp0)
    )

S_Q = S_from_Efunc(lambda d: -np.cos(d))
# For local model, use the sign-model E(Δ) computed via sampling
S_L = S_from_Efunc(lambda d: E_local_sign(abs(d), nsamp=200000))

# ----------------------------
# Plot
# ----------------------------
fig = plt.figure(figsize=(13, 7.6))
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.9], hspace=0.25, wspace=0.18)

axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])
axE = fig.add_subplot(gs[1, :])

# Heatmaps (use default colormap)
imL = axL.imshow(
    H_L.T,  # transpose so axes align with X/Y
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    aspect="equal",
    interpolation="nearest",
)
axL.axvline(np.pi, linewidth=2)
axL.axhline(np.pi, linewidth=2)
axL.set_title(f"Local sign model (deterministic given λ)\nΔ={delta*deg:.0f}°  E≈{E_L:+.3f}", fontsize=12)
axL.set_xlabel("x (shifted θA)")
axL.set_ylabel("y (shifted θB)")
axL.set_xticks([0, np.pi, 2*np.pi], ["0", "π", "2π"])
axL.set_yticks([0, np.pi, 2*np.pi], ["0", "π", "2π"])

imR = axR.imshow(
    rho_Q.T,
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    aspect="equal",
    interpolation="nearest",
)
axR.axvline(np.pi, linewidth=2)
axR.axhline(np.pi, linewidth=2)
axR.set_title(f'“Cheating” torus density (depends on settings)\nΔ={delta*deg:.0f}°  E={E_Q:+.3f}', fontsize=12)
axR.set_xlabel("x (shifted θA)")
axR.set_ylabel("y (shifted θB)")
axR.set_xticks([0, np.pi, 2*np.pi], ["0", "π", "2π"])
axR.set_yticks([0, np.pi, 2*np.pi], ["0", "π", "2π"])

# Annotate quadrant probabilities on both
centers = {
    "++": (np.pi/2, np.pi/2),
    "+-": (3*np.pi/2, np.pi/2),
    "-+": (np.pi/2, 3*np.pi/2),
    "--": (3*np.pi/2, 3*np.pi/2),
}

# Left quadrant estimates from density
valsL = {"++": Ppp_L, "+-": Ppm_L, "-+": Pmp_L, "--": Pmm_L}
for key, (cx, cy) in centers.items():
    axL.text(cx, cy, f"{key}\n{valsL[key]:.3f}", ha="center", va="center",
             fontsize=10, bbox=dict(boxstyle="round", alpha=0.75))

# Right quadrant exact targets
valsR = {"++": Ppp_Q, "+-": Ppm_Q, "-+": Pmp_Q, "--": Pmm_Q}
for key, (cx, cy) in centers.items():
    axR.text(cx, cy, f"{key}\n{valsR[key]:.3f}", ha="center", va="center",
             fontsize=10, bbox=dict(boxstyle="round", alpha=0.75))

# E(Δ) curve comparison
axE.plot(deltas_L*deg, Ecurve_L, lw=2, label="Local sign model (sampled)")
axE.plot(deltas*deg, Ecurve_Q, lw=2, label="Quantum singlet: E(Δ) = −cosΔ")
axE.axhline(0, linewidth=1)
axE.grid(True, alpha=0.35)
axE.set_xlim(0, 180)
axE.set_ylim(-1.05, 1.05)
axE.set_xlabel("Δ (degrees)")
axE.set_ylabel("E(Δ)")
axE.set_title(f"Correlation laws + CHSH S at (a=0°,a′=90°,b=±45°)\n"
              f"Local sign: S≈{S_L:+.3f}   |   Quantum: S={S_Q:+.3f}",
              fontsize=12)
axE.legend(frameon=False, fontsize=10, loc="lower right")

plt.show()

# Print a small numeric summary (also helpful)
print("Δ=45°")
print(f"  Local sign model: E≈{E_L:+.4f} (from λ), E≈{E_L_from_quads:+.4f} (from torus density approx)")
print(f"    Quadrants ~ ++ {Ppp_L:.4f}, +- {Ppm_L:.4f}, -+ {Pmp_L:.4f}, -- {Pmm_L:.4f}")
print(f"  Quantum target  : E ={E_Q:+.4f} (exact), E≈{E_Q_from_quads:+.4f} (from torus density)")
print(f"    Quadrants   = ++ {Ppp_Q:.4f}, +- {Ppm_Q:.4f}, -+ {Pmp_Q:.4f}, -- {Pmm_Q:.4f}")
